In [ ]:
                             #### CONTEXTO, CREACIÓN DE IMAGENES, ETCETERA ####

In [2]:
#Setup para confirmar directorio de trabajo
import os
os.getcwd()

'/home/sagemaker-user/demand-forecasting-model'

In [3]:
# Validamos estrcutura del repo de manera adecuada
!ls
!ls processing
!ls processing/container

LICENSE    data       processing      sm_processing_byoc.ipynb
README.md  docs       pyproject.toml  src
artifacts  notebooks  sagemaker       uv.lock
code  container
Dockerfile


In [4]:
# Construcción de la imagen Docker para el processing job
image_name = "sagemaker-processing-byoc"
image_tag = "latest"
!cd processing/container && docker build --network sagemaker -t {image_name}:{image_tag} .

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.

Sending build context to Docker daemon  2.048kB
Step 1/5 : FROM python:3.11-slim
3.11-slim: Pulling from library/python

1dee3f47: Pulling fs layer 
007a5b45: Pulling fs layer 
92e2619d: Pulling fs layer 
Digest: sha256:9358444059ed78e2975ada2c189f1c1a3144a5dab6f35bff8c981afb38946634
Status: Downloaded newer image for python:3.11-slim
 ---> e67db9b14d09
Step 2/5 : WORKDIR /opt/ml/processing/code
 ---> Running in 7c53fda561d3
 ---> Removed intermediate container 7c53fda561d3
 ---> e833471f28fe
Step 3/5 : RUN pip install --no-cache-dir scikit-learn pandas numpy
 ---> Running in 432a0a59b30e
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 15.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 255.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# Variables de AWS y ECR
import boto3
import subprocess

aws_account_id = subprocess.check_output(
    "aws sts get-caller-identity --query Account --output text",
    shell=True,
    text=True
).strip()

aws_region = boto3.Session().region_name

image_name = "sagemaker-processing-byoc"
image_tag = "latest"
ecr_uri = f"{aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com/{image_name}:{image_tag}"

print("AWS account:", aws_account_id)
print("AWS region:", aws_region)
print("ECR URI:", ecr_uri)

AWS account: 988261566883
AWS region: us-east-1
ECR URI: 988261566883.dkr.ecr.us-east-1.amazonaws.com/sagemaker-processing-byoc:latest


In [7]:
# Crear repositorio en ECR si no existe
!aws ecr describe-repositories --repository-names {image_name} --region {aws_region} >/dev/null 2>&1 || \
aws ecr create-repository --repository-name {image_name} --region {aws_region}

{
    "repository": {
        "repositoryArn": "arn:aws:ecr:us-east-1:988261566883:repository/sagemaker-processing-byoc",
        "registryId": "988261566883",
        "repositoryName": "sagemaker-processing-byoc",
        "repositoryUri": "988261566883.dkr.ecr.us-east-1.amazonaws.com/sagemaker-processing-byoc",
        "createdAt": "2026-03-22T01:47:49.985000+00:00",
        "imageTagMutability": "MUTABLE",
        "imageScanningConfiguration": {
            "scanOnPush": false
        },
        "encryptionConfiguration": {
            "encryptionType": "AES256"
        }
    }
}


In [8]:
# Login a ECR
!aws ecr get-login-password --region {aws_region} | \
docker login --username AWS --password-stdin {aws_account_id}.dkr.ecr.{aws_region}.amazonaws.com

WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

Login Succeeded


In [9]:
# Tag de la imagen para ECR
!docker tag {image_name}:{image_tag} {ecr_uri}

In [10]:
# Push de la imagen a ECR
!docker push {ecr_uri}

The push refers to repository [988261566883.dkr.ecr.us-east-1.amazonaws.com/sagemaker-processing-byoc]

3304b9d4: Preparing 
df4487e6: Preparing 
a0657a00: Preparing 
37810a00: Preparing 
95d9eb1d: Preparing 
latest: digest: sha256:700cf72d12462636b5a53fe157bd4db872eef510b223d0e4c14cc490c57ab535 size: 1579


In [ ]:
                                                 #### PROCESSING JOB #### 

In [11]:
import sagemaker
import boto3

sess = sagemaker.Session()
role = sagemaker.get_execution_role()

bucket = sess.default_bucket()

prefix = "demand-forecasting/processing-byoc"

raw_s3_uri = f"s3://{bucket}/{prefix}/raw"
processed_s3_uri = f"s3://{bucket}/{prefix}/processed"

print("Bucket:", bucket)
print("Raw:", raw_s3_uri)
print("Processed:", processed_s3_uri)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Bucket: sagemaker-us-east-1-988261566883
Raw: s3://sagemaker-us-east-1-988261566883/demand-forecasting/processing-byoc/raw
Processed: s3://sagemaker-us-east-1-988261566883/demand-forecasting/processing-byoc/processed


In [12]:
# Verificación de archivos en la ruta del repo previo a subirlos a S3
from pathlib import Path

raw_data_dir = Path("data/raw")

required_files = [
    "sales_train.csv",
    "items.csv",
    "item_categories.csv",
    "shops.csv",
]

print("Ruta evaluada:", raw_data_dir.resolve())

for f in required_files:
    file_path = raw_data_dir / f
    print(f, "->", file_path.exists())

Ruta evaluada: /home/sagemaker-user/demand-forecasting-model/data/raw
sales_train.csv -> True
items.csv -> True
item_categories.csv -> True
shops.csv -> True


In [13]:
# Subir archivos crudos a S3 para el Processing Job
for file_name in required_files:
    local_file = raw_data_dir / file_name
    
    s3_uri = sess.upload_data(
        path=str(local_file),
        bucket=bucket,
        key_prefix=f"{prefix}/raw"
    )
    
    print(f"{file_name} -> {s3_uri}")

sales_train.csv -> s3://sagemaker-us-east-1-988261566883/demand-forecasting/processing-byoc/raw/sales_train.csv
items.csv -> s3://sagemaker-us-east-1-988261566883/demand-forecasting/processing-byoc/raw/items.csv
item_categories.csv -> s3://sagemaker-us-east-1-988261566883/demand-forecasting/processing-byoc/raw/item_categories.csv
shops.csv -> s3://sagemaker-us-east-1-988261566883/demand-forecasting/processing-byoc/raw/shops.csv


In [14]:
# Crear ScriptProcessor para ejecutar el contenedor BYOC
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

script_processor = ScriptProcessor(
    image_uri=ecr_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=sess,
)

script_processor

In [15]:
# Ejecutar el Processing Job con preprocess.py
script_processor.run(
    code="processing/code/preprocess.py",
    inputs=[
        ProcessingInput(
            source=raw_s3_uri,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output",
            destination=processed_s3_uri
        )
    ],
    wait=True,
    logs=True
)

INFO:sagemaker:Creating processing-job with name sagemaker-processing-byoc-2026-03-22-02-08-41-206


........2026-03-22 02:09:55,380 - INFO - __main__ - Starting preprocessing job
2026-03-22 02:09:55,380 - INFO - __main__ - Input dir: /opt/ml/processing/input
2026-03-22 02:09:55,381 - INFO - __main__ - Output dir: /opt/ml/processing/output
2026-03-22 02:09:55,381 - INFO - __main__ - Loading raw data from /opt/ml/processing/input
2026-03-22 02:09:57,311 - INFO - __main__ - Raw data loaded: 2935849 rows, 10 columns
2026-03-22 02:09:57,312 - INFO - __main__ - Starting data cleaning
2026-03-22 02:09:59,994 - INFO - __main__ - Removed 6 duplicate rows
2026-03-22 02:09:59,995 - INFO - __main__ - Data cleaning completed: 2935843 rows
2026-03-22 02:09:59,995 - INFO - __main__ - Starting feature engineering
2026-03-22 02:10:00,985 - INFO - __main__ - Feature engineering completed: 1609124 rows, 7 columns
2026-03-22 02:10:05,655 - INFO - __main__ - Prepared data saved to /opt/ml/processing/output/sales_prep.csv
2026-03-22 02:10:05,655 - INFO - __main__ - Preprocessing job completed successfully

In [16]:
# Leer las primeras filas del archivo transformado desde S3
import pandas as pd

output_file_s3 = f"{processed_s3_uri}/sales_prep.csv"
print("Archivo de salida:", output_file_s3)

df_out = pd.read_csv(output_file_s3)
df_out.head()

Archivo de salida: s3://sagemaker-us-east-1-988261566883/demand-forecasting/processing-byoc/processed/sales_prep.csv


,month,date_block_num,shop_id,item_id,item_category_id,item_cnt_month,avg_price
0,1,0,0,51,57,2.0,128.5
1,1,0,0,61,43,1.0,195.0
2,1,0,0,75,40,1.0,76.0
3,1,0,0,88,40,1.0,76.0
4,1,0,0,95,40,1.0,193.0
